# EXP 1 — 형태소 분석: 4가지 방법 비교 (Demo Notebook)

- 본 논문 실험 1(형태소 분석)의 SFT/추론 파이프라인 재현 예시임
- 실제 논문에 사용된 데이터(모두의 말뭉치)는 라이선스 문제로 재배포 불가
- 국립국어원이 공식 공개한 예시 문장 1개를 100개로 복제한 데모 데이터 사용함

> ⚠️ 주의: 본 노트북은 "논문 결과 재현"이 아니라 "구현 투명성 제공"이 목적임.
> 데이터가 동일 문장의 반복이라 정확도 수치 자체는 무의미함. **파이프라인 실제 동작 확인**이 용도임.

> ⚠️ 실행 전 필수: NVIDIA 드라이버·CUDA 등 OS 레벨 환경 설정은
> 먼저 `readme/README.md` 참고해 완료할 것. Python 패키지 설치는
> 아래 "패키지 설치" 셀에서 진행함.

## 이 노트북에서 확인 가능한 4가지 결과

| # | 방법 | 설명 | 필요 자원 |
|---|---|---|---|
| 1 | **Only Inference (Gemma)** | 로컬 Gemma 모델 + few-shot 프롬프팅만 사용 (SFT 없음) | GPU + HuggingFace 토큰 |
| 2 | **Only Inference (GPT)** | OpenAI GPT API + few-shot 프롬프팅 | OpenAI API 키 (GPU 불필요) |
| 3 | **SFT (Gemma)** | 로컬 Gemma 모델을 QLoRA로 SFT한 뒤 추론 | GPU + HuggingFace 토큰 |
| 4 | **Kiwi** | 규칙/통계 기반 형태소 분석기 (LLM이 아닌 베이스라인) | 없음 (CPU만) |

## 폴더 구조 (고정)
```
./readme/    — 환경 설정 안내 (README.md)
./notebook/  — 이 노트북 파일
./data/      — nikl_mp_v1_1_repeated_100_sample.json
```
본 노트북은 `./notebook/` 폴더 안에 있다고 가정함. 데이터 파일은
`../data/nikl_mp_v1_1_repeated_100_sample.json` 상대경로로 탐색함.

## 환경 변수 (필요한 것만 설정하면 해당 섹션 실행 가능)
- `HF_TOKEN` — Gemma 모델 다운로드용 (결과 1, 3)
- `OPENAI_API_KEY` — GPT API용 (결과 2). `.env` 파일 사용 가능.
- 자세한 설정 방법은 `readme/README.md` 참고.

각 섹션(결과 1~4)은 서로 독립적으로 실행 가능함.
GPU 없으면 결과 2(GPT), 결과 4(Kiwi)만 실행해도 됨.


## 패키지 설치

- 아래 셀은 본 노트북 실행에 필요한 패키지를 설치함
- 버전은 실제 실험 환경(Python 3.12.3 / CUDA 12.8)에서 검증된 버전으로 고정함. GPU VRAM은 24GB 이상 권장함 (자세한 내용은 readme/README.md 참고)
- 다른 CUDA 버전 환경이면 torch 설치 줄만 본인 환경에 맞게 수정하면 됨
  (자세한 환경별 안내는 `readme/README.md` 참고)
- GPU 없이 결과 2(GPT)·결과 4(Kiwi)만 실행할 계획이면, torch/Gemma 관련 설치 셀은 건너뛰어도 됨


In [1]:
# ── Cell 0-a. 패키지 설치 ────────────────────────────────────
# 이미 설치돼 있으면 '--quiet' 옵션으로 출력만 억제하고 그대로 통과함.

# ── PyTorch (CUDA 12.8 빌드) — 결과 1, 3(Gemma)에 필요 ─────────
# PyPI 기본 인덱스에는 +cu128 빌드가 없어 PyTorch 전용 인덱스에서 설치함.
!pip install torch==2.10.0+cu128 torchaudio==2.11.0+cu128 torchvision==0.25.0+cu128 --index-url https://download.pytorch.org/whl/cu128 --quiet

# ── Gemma 추론/SFT용 — 결과 1, 3 ────────────────────────────────
!pip install transformers==5.5.4 peft==0.19.1 trl==1.2.0 bitsandbytes==0.49.2 accelerate==1.13.0 datasets==4.8.4 --quiet

# ── Kiwi 형태소 분석기 — 결과 4 ─────────────────────────────────
!pip install kiwipiepy==0.23.1 kiwipiepy_model==0.23.0 --quiet

# ── GPT API — 결과 2 ────────────────────────────────────────────
!pip install openai==2.36.0 python-dotenv==1.2.2 --quiet

# ── 공통 데이터 처리·집계 ────────────────────────────────────────
!pip install pandas==3.0.2 numpy==2.4.3 tqdm==4.67.3 openpyxl==3.1.5 --quiet

print("패키지 설치 완료")



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
패키지 설치 완료


In [2]:
# ── Cell 0-b. 설치 확인 ──────────────────────────────────────
# CUDA_VISIBLE_DEVICES는 반드시 'import torch' 이전에 설정해야 효과 있음.
# (torch가 CUDA를 한 번 초기화하면 이후 이 환경변수를 바꿔도 무시됨.)
# 본 노트북은 GPU 0번 하나만 사용하도록 프로세스 시작 시점에 고정함.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

try:
    import torch
    print(f"torch 버전:        {torch.__version__}")
    print(f"CUDA available:     {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU 개수(프로세스 기준): {torch.cuda.device_count()}")  # CUDA_VISIBLE_DEVICES 덕분에 1이어야 정상
        for i in range(torch.cuda.device_count()):
            print(f"  [{i}] {torch.cuda.get_device_name(i)}  "
                  f"({torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB)")
    else:
        print("GPU 미인식 — CPU로 진행됩니다. 결과 1·3(Gemma) 섹션은 매우 느리거나 동작하지 않을 수 있습니다.")
except ImportError:
    print("torch가 설치되지 않았습니다. 결과 2(GPT)·결과 4(Kiwi)만 실행할 계획이면 무시해도 됩니다.")


torch 버전:        2.10.0+cu128
CUDA available:     True
GPU 개수(프로세스 기준): 1
  [0] NVIDIA GeForce RTX 5090  (31.4 GB)


## 실행 전 경로 확인

- 노트북을 여는 프로그램(Jupyter/Docker 등)에 따라 커널의 작업 디렉토리가
  `./notebook/`이 아닐 수 있음
- 아래 **Cell 0-0**을 먼저 실행해 데이터 파일을 정상적으로 찾는지 확인할 것
- 못 찾으면 셀 출력의 안내를 따라 `NIKL_JSON_PATH`를 직접 수정하면 됨


In [3]:
# ── Cell 0-0. 작업 디렉토리 · 데이터 파일 경로 진단 ──────────
import os

print(f"[현재 작업 디렉토리]\n  {os.getcwd()}")

expected = "../data/nikl_mp_v1_1_repeated_100_sample.json"
if os.path.exists(expected):
    print(f"\n✅ 데이터 파일을 정상적으로 찾았습니다: {expected}")
else:
    print(f"\n❌ '{expected}' 경로에서 데이터 파일을 찾지 못했습니다.")
    print("   폴더 구조(./readme, ./notebook, ./data)를 확인하거나,")
    print("   아래 '공통 설정' 셀에서 NIKL_JSON_PATH를 직접 수정하세요.")


[현재 작업 디렉토리]
  /home/jth-home/Downloads/notebook

✅ 데이터 파일을 정상적으로 찾았습니다: ../data/nikl_mp_v1_1_repeated_100_sample.json


In [4]:
# ── Cell 0-1. 공통 설정 ──────────────────────────────────────
# 본 노트북 전체에서 공유하는 경로/시드 상수.
# 실제 논문 실험(exp1_morpheme_*_v4.ipynb)과 동일한 시드 정책 유지함.
#   - SPLIT_SEED = 22 : 데이터 분할/서브샘플링용
#   - SHOT_SEED  = 0  : few-shot 예시 샘플링용
#
# 패키지 설치 등 환경 설정은 이미 끝났다고 가정함 (readme/README.md 참고).

import json  # 데이터 변환·로드 전 구간에서 공통으로 사용
import os
# 고정 폴더 구조(./readme, ./notebook, ./data) 기준 상대경로.
# Cell 0-0에서 파일을 못 찾았으면, 아래 NIKL_JSON_PATH를 절대경로로 직접 수정할 것.
NIKL_JSON_PATH  = "../data/nikl_mp_v1_1_repeated_100_sample.json"
DEMO_JSONL_PATH = "../data/written_full_demo.jsonl"  # 변환 결과도 data 폴더에 저장

SPLIT_SEED = 22
SHOT_SEED  = 0
DOMAIN     = "written"

# ── 파일 존재 확인 ────────────────────────────────────────────
if not os.path.exists(NIKL_JSON_PATH):
    print(f"'{NIKL_JSON_PATH}' 없음 → 상위 폴더에서 자동 탐색 시도...")
    found = None
    for root, dirs, files in os.walk(".."):
        if "nikl_mp_v1_1_repeated_100_sample.json" in files:
            found = os.path.join(root, "nikl_mp_v1_1_repeated_100_sample.json")
            break
    if found:
        NIKL_JSON_PATH = found
        print(f"자동으로 찾음: {NIKL_JSON_PATH}")
    else:
        print("자동 탐색 실패. NIKL_JSON_PATH를 절대경로로 직접 지정해야 합니다.")

print(f"\nNIKL_JSON_PATH:  {NIKL_JSON_PATH}  (존재: {os.path.exists(NIKL_JSON_PATH)})")
print(f"DEMO_JSONL_PATH: {DEMO_JSONL_PATH}")



NIKL_JSON_PATH:  ../data/nikl_mp_v1_1_repeated_100_sample.json  (존재: True)
DEMO_JSONL_PATH: ../data/written_full_demo.jsonl


## 데이터 준비: NIKL JSON → SFT용 JSONL 변환

- 실제 실험에서는 모두의 말뭉치 원본 형태 분석 JSON을 아래 `messages` 포맷의
  JSONL로 변환해 사용함. 본 셀은 그 변환 과정을 동일한 방식으로 재현함.

- `system`: 형태소 분석 태그셋 + 분절 규칙 안내 (모든 샘플에 동일하게 적용되는 고정 instruction)
- `user`: 분석 대상 문장 (NIKL의 `sentence.form`)
- `assistant`: 정답 형태소 분석 결과 (`형태소/태그 형태소/태그 ...` 형식)


In [5]:
# ── Cell 1. 시스템 메시지(instruction) 정의 ───────────────────
# 실제 실험(exp1_sample_written_full.jsonl)에서 사용한 것과 동일한
# 태그 목록 + 분절 규칙 instruction. 모든 샘플의 system 메시지로 재사용됨.

SYSTEM_TEXT = """너는 한국어 형태소 분석 전문가다. 아래의 태그 목록과 분절 규칙을 참고해 문장에 대한 형태소 분석을 수행하라.

## 태그 세트
[체언] NNG 일반명사 | NNP 고유명사 | NNB 의존명사 | NP 대명사 | NR 수사
[용언] VV 동사 | VA 형용사 | VX 보조용언 | VCP 긍정지정사 | VCN 부정지정사
[수식언] MMA 성상관형사 | MMD 지시관형사 | MMN 수관형사 | MAG 일반부사 | MAJ 접속부사
[독립언] IC 감탄사
[관계언] JKS 주격 | JKC 보격 | JKG 관형격 | JKO 목적격 | JKB 부사격 | JKV 호격 | JKQ 인용격 | JX 보조사 | JC 접속조사
[의존형태] EP 선어말어미 | EF 종결어미 | EC 연결어미 | ETN 명사형전성어미 | ETM 관형형전성어미 | XPN 체언접두사 | XSN 명사파생접미사 | XSV 동사파생접미사 | XSA 형용사파생접미사 | XR 어근
[기호] SF 마침표/물음표/느낌표 | SP 쉼표류 | SS 따옴표/괄호 | SE 줄임표 | SO 붙임표 | SW 기타기호
[기타] SL 외국어 | SH 한자 | SN 숫자 | NA 분석불능 | NF 명사추정 | NV 용언추정

## 분절 규칙
반드시 형태소 단위로 분리하라. 어절 단위로 묶어서 출력하지 말라.
본 분석은 엄밀히 말해 '형태소' 차원이 아닌 '형태' 차원의 분석이므로 이형태를 최대한 반영한다.
예) 잡아서 → 잡/VV 아서/EC | 먹어서 → 먹/VV 어서/EC

## 출력 형식
형태소/태그 형태소/태그 ...
예) 만나/VV 았/EP 습니다/EF ./SF
    """

print(f"system 메시지 길이: {len(SYSTEM_TEXT)}자")
print(SYSTEM_TEXT[:80], "...")


system 메시지 길이: 799자
너는 한국어 형태소 분석 전문가다. 아래의 태그 목록과 분절 규칙을 참고해 문장에 대한 형태소 분석을 수행하라.

## 태그 세트
[체언] NN ...


In [6]:
# ── Cell 2. NIKL JSON → (form, tag) 리스트 변환 함수 ───────────
# NIKL 형태 분석 포맷: document > sentence > morpheme (word_id, position 순)
# position 순 정렬 시 실제 형태소 등장 순서가 됨.

def nikl_sentence_to_pairs(sentence_obj: dict) -> list:
    """
    NIKL 'sentence' 객체 하나를 (form, tag) 튜플 리스트로 변환.
    morpheme은 (word_id, position) 기준으로 정렬해 원래 어절/음절 순서를 보존한다.
    """
    morphemes = sorted(
        sentence_obj["morpheme"],
        key=lambda m: (m["word_id"], m["position"]),
    )
    return [(m["form"], m["label"]) for m in morphemes]


def nikl_json_to_messages(nikl_json_path: str, system_text: str) -> list:
    """
    NIKL 형태 분석 JSON 파일 전체를 읽어 SFT용 messages 리스트로 변환.
    문서(document) > 문장(sentence) 단위로 순회하며 하나의 문장 = 하나의 샘플.

    반환:
        [{"messages": [{"role": "system", ...}, {"role": "user", ...}, {"role": "assistant", ...}]}, ...]
    """
    with open(nikl_json_path, encoding="utf-8") as f:
        raw = json.load(f)

    converted = []
    for document in raw["document"]:
        for sentence_obj in document["sentence"]:
            pairs = nikl_sentence_to_pairs(sentence_obj)
            if not pairs:
                continue  # 형태소 정보 없는 문장은 건너뜀

            user_text      = sentence_obj["form"]
            assistant_text = " ".join(f"{form}/{tag}" for form, tag in pairs)

            converted.append({
                "messages": [
                    {"role": "system",    "content": system_text},
                    {"role": "user",      "content": user_text},
                    {"role": "assistant", "content": assistant_text},
                ]
            })
    return converted


In [7]:
# ── Cell 3. 변환 실행 + JSONL 저장 ──────────────────────────────
# 실제 실험 코드가 원본 NIKL JSON을 jsonl로 변환해 사용했던 절차와 동일함.
# 데모 데이터는 문장 1개가 100번 반복된 파일이므로, 변환 결과도
# 동일한 messages 100개가 됨 (내용은 전부 같음 — 동작 확인용).

if not os.path.exists(NIKL_JSON_PATH):
    raise FileNotFoundError(
        f"'{NIKL_JSON_PATH}' 파일을 찾을 수 없습니다.\n"
        f"  - 현재 작업 디렉토리: {os.getcwd()}\n"
        f"  - 위 '공통 설정' 셀에서 NIKL_JSON_PATH를 파일의 절대경로로 직접 지정하거나,\n"
        f"    파일을 현재 작업 디렉토리로 옮긴 뒤 다시 실행하세요.\n"
        f"    (자세한 안내는 노트북 상단 '실행 전 환경 설정' 섹션 참고)"
    )

converted_samples = nikl_json_to_messages(NIKL_JSON_PATH, SYSTEM_TEXT)
print(f"변환된 샘플 수: {len(converted_samples)}")

with open(DEMO_JSONL_PATH, "w", encoding="utf-8") as f:
    for sample in converted_samples:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

print(f"JSONL 저장 완료: {DEMO_JSONL_PATH}")

# ── 변환 결과 확인 ────────────────────────────────────────────
print("\n[샘플 0 확인]")
print(f"  user:      {converted_samples[0]['messages'][1]['content']}")
print(f"  assistant: {converted_samples[0]['messages'][2]['content'][:80]}...")


변환된 샘플 수: 100
JSONL 저장 완료: ../data/written_full_demo.jsonl

[샘플 0 확인]
  user:      [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  assistant: [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개...


## 공통 함수 정의

- 아래 함수들은 원 실험 노트북(`exp1_morpheme_inference_v4.ipynb`,
  `exp1_morpheme_SFT_v4.ipynb`, `exp1_morpheme_inference_GPT_v3.ipynb`,
  `exp1_morpheme_kiwi_v3.ipynb`)에서 동일하게 사용하던 함수를 그대로 가져온 것임
- 4가지 방법 모두 **같은 평가 기준**으로 비교해야 공정한 비교가 되므로,
  본 노트북에서도 하나의 정의를 4개 섹션이 공유함


In [8]:
# ── Cell 4. 데이터 로드 + 필터링 + 분할 ─────────────────────
# 원본 실험과 동일한 절차: jsonl 로드 → 토크나이저로 assistant 토큰 수 계산
# → p95 필터링(극단적 outlier 제거) → seed 고정 분할(8:1:1)
#
# [DEMO 노트북 주의사항]
# 데모 데이터는 동일 문장 100개 반복이라 토큰 수 분포가 전부 동일함.
# 따라서 p95 필터링은 사실상 아무것도 제거하지 않으나,
# 실제 실험과 동일한 코드 경로를 그대로 보여주기 위해 남겨둠.

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def get_system_text(sample):
    for m in sample["messages"]:
        if m["role"] == "system": return m["content"]
    return ""

def get_user_text(sample):
    for m in sample["messages"]:
        if m["role"] == "user": return m["content"]
    return ""

def get_assistant_text(sample):
    for m in sample["messages"]:
        if m["role"] == "assistant": return m["content"]
    return ""


def load_filter_split(jsonl_path, tokenizer, seed=SPLIT_SEED):
    """
    jsonl 로드 → assistant 토큰 수 기준 p95 필터링 → seed 고정 8:1:1 분할.
    tokenizer가 없는 섹션(GPT, Kiwi)에서는 공백 기준 토큰 수로 근사한다.
    """
    import random
    import numpy as np

    samples = load_jsonl(jsonl_path)
    print(f"원본 샘플 수: {len(samples)}")

    if tokenizer is not None:
        token_lens = [len(tokenizer.encode(get_assistant_text(s))) for s in samples]
    else:
        token_lens = [len(get_assistant_text(s).split()) for s in samples]

    p95 = np.percentile(token_lens, 95)
    filtered = [s for s, l in zip(samples, token_lens) if l <= p95]
    print(f"필터링 후: {len(filtered)}개 (p95={p95:.1f} 기준)")

    random.seed(seed)
    shuffled = filtered.copy()
    random.shuffle(shuffled)

    n       = len(shuffled)
    n_train = max(1, int(n * 0.8))
    n_val   = max(1, int(n * 0.1))

    train_samples = shuffled[:n_train]
    val_samples   = shuffled[n_train:n_train + n_val]
    test_samples  = shuffled[n_train + n_val:] or shuffled[-1:]  # 최소 1개 보장

    print(f"train: {len(train_samples)} | val: {len(val_samples)} | test: {len(test_samples)}")
    return train_samples, val_samples, test_samples


In [9]:
# ── Cell 5. 평가 함수 정의 ────────────────────────────────────
# [평가 방식] (exp1_morpheme_inference_v4.ipynb Cell 6과 동일)
# Strict  — 문장 단위. 하나라도 틀리면 0.
#   1) seg_strict:  모든 form이 일치하면 1, 아니면 0
#   2) full_strict: 모든 form+label이 일치하면 1, 아니면 0
#
# Degree  — LCS 기반. 순서 유지. 누락 허용. 부분 점수 허용.
#   1) seg_degree:  LCS(gold forms, pred forms) / gold 형태소 수
#   2) full_degree: LCS(gold pairs, pred pairs) / gold 형태소 수
#
# [LCS를 쓰는 이유]
# - Counter 방식은 순서를 무시함 → 중복 형태소 있을 때 오류
# - LCS는 순서를 유지하면서 누락만 허용함 → 중복 형태소도 정확히 처리
#
# [Strict vs. Degree]
# - Strict: 완전 정답률. 실용적 기준.
# - Degree: 부분 정답률. 모델 성능을 연속적으로 측정.
# - Strict Accuracy ≤ Degree Accuracy 항상 성립함.

import re

def parse_assistant_output(assistant_str):
    """
    '국수/NNG 넣/VV 구/EC 그렇/VA 죠/EF'
    → [('국수','NNG'), ('넣','VV'), ('구','EC'), ('그렇','VA'), ('죠','EF')]
    파싱 실패한 토큰은 (tok, '??')로 대체
    전체가 빈 경우만 [] 반환
    """
    try:
        tokens = assistant_str.strip().split()
        result = []
        for tok in tokens:
            if "/" in tok:
                form, label = tok.rsplit("/", 1)
                result.append((form, label))
            else:
                result.append((tok, "??"))
        return result if result else []
    except Exception:
        return []

def normalize_pred(pred_raw):
    """
    모델 출력에서 형태소/태그 패턴이 있는 줄 추출
    앞뒤 불필요한 텍스트 제거
    """
    for line in pred_raw.strip().splitlines():
        line = line.strip().strip("`")
        if re.search(r"\S+/[A-Z]+", line):
            return line
    return pred_raw.strip()

def lcs_length(seq1, seq2):
    """
    LCS(Longest Common Subsequence) 길이 계산
    순서를 유지하면서 공통으로 매칭되는 최대 원소 수 반환
    시간복잡도: O(n*m)
    """
    n, m = len(seq1), len(seq2)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if seq1[i-1] == seq2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[n][m]

def compute_metrics(gold_pairs, pred_pairs):
    """
    gold_pairs: [(form, label), ...]  — 정답
    pred_pairs: [(form, label), ...]  — 모델 예측

    반환값:
        seg_strict:  모든 form 일치 여부 (0 or 1)
        full_strict: 모든 form+label 일치 여부 (0 or 1)
        seg_lcs:     LCS(gold forms, pred forms) 길이 (degree 분자)
        full_lcs:    LCS(gold pairs, pred pairs) 길이 (degree 분자)
        total:       gold 형태소 수 (degree 분모)
        seg_degree:  seg_lcs / total
        full_degree: full_lcs / total
    """
    if not gold_pairs:
        return {
            "seg_strict": 0, "full_strict": 0,
            "seg_lcs": 0, "full_lcs": 0,
            "total": 0,
            "seg_degree": 0.0, "full_degree": 0.0,
        }

    total = len(gold_pairs)

    # ── Strict: 위치 대응 ─────────────────────────────────────
    if len(pred_pairs) != len(gold_pairs):
        seg_strict  = 0
        full_strict = 0
    else:
        seg_strict  = 1 if all(gf == pf for (gf, _), (pf, _) in zip(gold_pairs, pred_pairs)) else 0
        full_strict = 1 if gold_pairs == pred_pairs else 0

    # ── Degree: LCS 방식 ──────────────────────────────────────
    gold_forms = [gf for gf, _ in gold_pairs]
    pred_forms = [pf for pf, _ in pred_pairs]

    seg_lcs  = lcs_length(gold_forms, pred_forms)  # form만 비교 (label 무시)
    full_lcs = lcs_length(gold_pairs, pred_pairs)  # form+label 모두 비교

    return {
        "seg_strict":  seg_strict,
        "full_strict": full_strict,
        "seg_lcs":     seg_lcs,
        "full_lcs":    full_lcs,
        "total":       total,
        "seg_degree":  seg_lcs  / total,
        "full_degree": full_lcs / total,
    }

def show_summary(label, results):
    """results 리스트를 받아 Strict/Degree 요약을 출력."""
    n = len(results)
    n_fail = sum(1 for r in results if not r.get("parse_success", True))
    seg_strict  = sum(r["seg_strict"]  for r in results) / n * 100
    full_strict = sum(r["full_strict"] for r in results) / n * 100
    seg_deg     = sum(r["seg_degree"]  for r in results) / n * 100
    full_deg    = sum(r["full_degree"] for r in results) / n * 100
    print(f"\n=== [{label}] 결과 요약 (n={n}) ===")
    print(f"  parse_fail: {n_fail} ({n_fail/n*100:.1f}%)")
    print(f"  [Strict]  seg: {seg_strict:.2f}%   full: {full_strict:.2f}%")
    print(f"  [Degree]  seg: {seg_deg:.2f}%   full: {full_deg:.2f}%")
    return {
        "method": label, "n": n, "parse_fail": n_fail,
        "seg_strict": round(seg_strict, 2), "full_strict": round(full_strict, 2),
        "seg_degree": round(seg_deg, 2), "full_degree": round(full_deg, 2),
    }


In [10]:
# ── Cell 6. 오류 분석 함수 정의 (Error Analysis) ────────────
# 오류를 두 가지 유형으로 처리함:
#
#   [A] 이형태 오류 (allomorph error)
#       어미 태그(EC/EF/EP/ETN/ETM)에서 label 일치 & form 불일치.
#       이형태 쌍의 완전한 대응 map 없이는 자동 분류가 불완전하므로
#       정량 집계는 하지 않고, 해당 케이스를 출력해 질적 분석에 활용함.
#
#   [B] 분절 오류 (segmentation error)
#       gold/pred 형태소 수(len)가 다른 경우.
#       자동 집계 + 예시 출력.
#
#   [C] 태그 오류는 집계하지 않음.
#       분절 밀림으로 인한 연쇄 오류와 구분 불가 → LCS Degree로 대체함.

ALLOMORPH_TAGS = {"EC", "EF", "EP", "ETN", "ETM"}

def get_allomorph_candidates(gold_pairs, pred_pairs):
    """
    어미 태그에서 label 일치 & form 불일치인 쌍을 반환.
    이형태 여부를 확정 판별하지 않고 후보만 수집.
    길이가 다르면 분절 오류이므로 빈 리스트 반환.

    반환:
        candidates: [(gold_pair, pred_pair), ...]
    """
    if not gold_pairs or not pred_pairs:
        return []
    if len(gold_pairs) != len(pred_pairs):
        return []

    candidates = []
    for (gf, gl), (pf, pl) in zip(gold_pairs, pred_pairs):
        if gl in ALLOMORPH_TAGS and gl == pl and gf != pf:
            candidates.append(((gf, gl), (pf, pl)))
    return candidates


def run_error_analysis(results, pred_key="pred_normalized"):
    """
    results 리스트를 받아 오류 분석 결과를 출력.
    [A]: 집계 없이 예시만 출력 (질적 분석용)
    [B]: 집계 + 예시 출력
    """
    total          = len(results)
    total_seg      = 0
    n_error_sents  = 0
    allomorph_candidates = []  # (sentence, gold_pair, pred_pair)
    seg_error_rows       = []

    for r in results:
        if r["full_strict"] == 1:
            continue
        n_error_sents += 1
        gold_pairs = parse_assistant_output(r["gold"])
        pred_pairs = parse_assistant_output(r[pred_key])

        # [B] 분절 오류
        if len(gold_pairs) != len(pred_pairs):
            total_seg += 1
            seg_error_rows.append(r)

        # [A] 이형태 후보 수집
        cands = get_allomorph_candidates(gold_pairs, pred_pairs)
        if cands:
            allomorph_candidates.extend(
                [(r["sentence"], g, p) for g, p in cands]
            )

    print("=" * 60)
    print("=== 오류 분석 ===")
    print(f"  오류 문장 수:  {n_error_sents} / {total} ({n_error_sents/total*100:.1f}%)")
    print(f"  [A] 이형태 후보: {len(allomorph_candidates)}건 (집계 안 함, 질적 분석용)")
    print(f"  [B] 분절 오류:   {total_seg}개 문장 ({total_seg/total*100:.1f}%)")


---
## 결과 1: Only Inference (Gemma)

- 로컬 Gemma 모델을 SFT 없이, few-shot 프롬프팅만으로 형태소 분석에 사용함
  (원 노트북: `exp1_morpheme_inference_v4.ipynb`)
- GPU + HuggingFace 토큰(`HF_TOKEN` 환경변수) 필요


In [11]:
# ── Cell 7. GPU 확인 + HF 로그인 + 캐시 경로 설정 ────────────
import os
# CUDA_VISIBLE_DEVICES는 Cell 0-b(import torch 이전)에서 이미 설정됨.
# 여기서 다시 설정해봐야 이미 초기화된 CUDA에는 반영 안 되므로, 값만 확인함.
assert os.environ.get("CUDA_VISIBLE_DEVICES") == "0", (
    "CUDA_VISIBLE_DEVICES가 예상과 다릅니다. Cell 0-b가 먼저 실행됐는지 확인하세요."
)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── HuggingFace 캐시 경로를 프로젝트 로컬 폴더로 고정 ───────────
os.environ.setdefault("HF_HOME", os.path.abspath("../.hf_cache"))
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
print(f"HF_HOME: {os.environ['HF_HOME']}")

import sys, torch
from huggingface_hub import login

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", DEVICE)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

hf_token = os.environ.get("HF_TOKEN")  # 환경변수 권장 (하드코딩 금지)
if hf_token:
    login(token=hf_token)
    print("HF login: OK")
else:
    print("HF_TOKEN not set. Gemma는 gated model이므로 로그인이 필요할 수 있음.")


HF_HOME: /home/jth-home/Downloads/.hf_cache
Python: /home/jth-home/venv/p312_test/bin/python
Torch: 2.10.0+cu128
CUDA available: True
Device: cuda:0
CUDA_VISIBLE_DEVICES: 0
HF_TOKEN not set. Gemma는 gated model이므로 로그인이 필요할 수 있음.


/home/jth-home/venv/p312_test/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# ── Cell 8. Settings + 모델/토크나이저 로드 (4-bit, inference용) ── (10분 이상 소요될 수 있음)
model_id = "google/gemma-4-E2B-it"

from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    dtype = torch.bfloat16
else:
    dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype,
    bnb_4bit_quant_storage=dtype,
)

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        dtype=dtype,
        device_map={"": 0} if torch.cuda.is_available() else "cpu",
        quantization_config=bnb_config if torch.cuda.is_available() else None,
    )
    print(f"모델 로드 완료: {model_id}")
except PermissionError as e:
    print(f"❌ 캐시 폴더 권한 오류: {e}")
    print("   → HF_HOME(Cell 7에서 설정) 폴더의 쓰기 권한을 확인하거나,")
    print("     readme/README.md의 'HuggingFace 캐시 권한 문제' 항목을 참고하세요.")
    raise
except OSError as e:
    if "PermissionError" in str(e) or "Check cache directory permissions" in str(e):
        print(f"❌ 캐시 폴더 권한 오류로 추정됨: {e}")
        print("   → readme/README.md의 'HuggingFace 캐시 권한 문제' 항목을 참고하세요.")
    raise


Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 1990.78it/s]


모델 로드 완료: google/gemma-4-E2B-it


In [13]:
# ── Cell 9. 데이터 로드 + 분할 (공통 함수 사용) ────────────────
train_samples, val_samples, test_samples = load_filter_split(
    DEMO_JSONL_PATH, tokenizer, seed=SPLIT_SEED
)

system_message = get_system_text(train_samples[0])
SYSTEM_MSG     = {"role": "system", "content": system_message}
print(f"\nsystem 토큰 수: {len(tokenizer.encode(system_message))}")


원본 샘플 수: 100
필터링 후: 100개 (p95=90.0 기준)
train: 80 | val: 10 | test: 10

system 토큰 수: 493


In [14]:
# ── Cell 10. Few-shot 예시 구성 ──────────────────────────────
# train set에서 N_SHOTS개 샘플을 few-shot 예시로 사용함.
# [DEMO] 데이터가 100개뿐이므로 min()으로 안전하게 개수를 맞춤.
import random

N_SHOTS = min(3, len(train_samples))
random.seed(SHOT_SEED)
few_shot_examples = random.sample(train_samples, N_SHOTS)

few_shot_messages = []
for ex in few_shot_examples:
    few_shot_messages.append({"role": "user",      "content": get_user_text(ex)})
    few_shot_messages.append({"role": "assistant", "content": get_assistant_text(ex)})

print(f"few-shot 예시 수: {len(few_shot_examples)}")
print(f"few-shot 예시: {few_shot_examples}")


few-shot 예시 수: 3
few-shot 예시: [{'messages': [{'role': 'system', 'content': "너는 한국어 형태소 분석 전문가다. 아래의 태그 목록과 분절 규칙을 참고해 문장에 대한 형태소 분석을 수행하라.\n\n## 태그 세트\n[체언] NNG 일반명사 | NNP 고유명사 | NNB 의존명사 | NP 대명사 | NR 수사\n[용언] VV 동사 | VA 형용사 | VX 보조용언 | VCP 긍정지정사 | VCN 부정지정사\n[수식언] MMA 성상관형사 | MMD 지시관형사 | MMN 수관형사 | MAG 일반부사 | MAJ 접속부사\n[독립언] IC 감탄사\n[관계언] JKS 주격 | JKC 보격 | JKG 관형격 | JKO 목적격 | JKB 부사격 | JKV 호격 | JKQ 인용격 | JX 보조사 | JC 접속조사\n[의존형태] EP 선어말어미 | EF 종결어미 | EC 연결어미 | ETN 명사형전성어미 | ETM 관형형전성어미 | XPN 체언접두사 | XSN 명사파생접미사 | XSV 동사파생접미사 | XSA 형용사파생접미사 | XR 어근\n[기호] SF 마침표/물음표/느낌표 | SP 쉼표류 | SS 따옴표/괄호 | SE 줄임표 | SO 붙임표 | SW 기타기호\n[기타] SL 외국어 | SH 한자 | SN 숫자 | NA 분석불능 | NF 명사추정 | NV 용언추정\n\n## 분절 규칙\n반드시 형태소 단위로 분리하라. 어절 단위로 묶어서 출력하지 말라.\n본 분석은 엄밀히 말해 '형태소' 차원이 아닌 '형태' 차원의 분석이므로 이형태를 최대한 반영한다.\n예) 잡아서 → 잡/VV 아서/EC | 먹어서 → 먹/VV 어서/EC\n\n## 출력 형식\n형태소/태그 형태소/태그 ...\n예) 만나/VV 았/EP 습니다/EF ./SF\n    "}, {'role': 'user', 'content': '[제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀'}, {'role': 'assistant', 

In [15]:
# ── Cell 11. 추론 파라미터 설정 ─────────────────────────────
import numpy as np

token_lengths = [len(tokenizer.encode(get_assistant_text(s))) for s in train_samples]

INFER_MAX_TOKENS  = int(np.max(token_lengths) * 1.5)
INFER_REP_PENALTY = 1.1
INFER_BATCH_SIZE  = min(4, len(test_samples))
SETTING           = "few_shot"

tokenizer.padding_side = "left"
tokenizer.pad_token    = tokenizer.eos_token

print(f"INFER_MAX_TOKENS:  {INFER_MAX_TOKENS}")
print(f"INFER_BATCH_SIZE:  {INFER_BATCH_SIZE}")


INFER_MAX_TOKENS:  135
INFER_BATCH_SIZE:  4


In [16]:
# ── Cell 12. 배치 추론 루프 ──────────────────────────────────
# padding_side="left" 설정으로 greedy 결과 동일 보장함.

results_gemma_inference = []

for batch_start in range(0, len(test_samples), INFER_BATCH_SIZE):
    batch_samples = test_samples[batch_start:batch_start + INFER_BATCH_SIZE]

    batch_messages = [
        [SYSTEM_MSG]
        + few_shot_messages
        + [{"role": "user", "content": get_user_text(s)}]
        for s in batch_samples
    ]

    inputs = tokenizer.apply_chat_template(
        batch_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        padding=True,
    ).to(DEVICE)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=INFER_MAX_TOKENS,
            do_sample=False,
            use_cache=True,
            top_p=None,
            top_k=None,
            repetition_penalty=INFER_REP_PENALTY,
        )

    for j, sample in enumerate(batch_samples):
        pred_tokens    = outputs[j][input_len:]
        truncated      = pred_tokens.shape[-1] >= INFER_MAX_TOKENS

        pred_raw        = tokenizer.decode(pred_tokens, skip_special_tokens=True).strip()
        pred_normalized = normalize_pred(pred_raw)
        gold_pairs      = parse_assistant_output(get_assistant_text(sample))
        pred_pairs      = parse_assistant_output(pred_normalized)
        m               = compute_metrics(gold_pairs, pred_pairs)

        results_gemma_inference.append({
            "sentence": get_user_text(sample), "gold": get_assistant_text(sample),
            "pred_raw": pred_raw, "pred_normalized": pred_normalized,
            "parse_success": len(pred_pairs) > 0, "truncated": truncated,
            **{k: m[k] for k in ["seg_strict", "full_strict", "seg_lcs", "full_lcs", "total", "seg_degree", "full_degree"]},
        })

print("완료!")
summary_gemma_inference = show_summary("Only Inference (Gemma)", results_gemma_inference)
run_error_analysis(results_gemma_inference)

# ── 실제 추론 예시 확인 ────────────────────────────────────────
# 집계된 숫자만으로는 모델이 실제로 어떻게 틀리는지 보기 어려우므로,
# gold(정답)와 pred(모델 출력)를 나란히 몇 개 직접 출력해 확인함.
N_EXAMPLES = min(3, len(results_gemma_inference))
print("\n" + "=" * 60)
print(f"=== 실제 추론 예시 ({N_EXAMPLES}개) ===")
print("=" * 60)
for i, r in enumerate(results_gemma_inference[:N_EXAMPLES]):
    mark = "✓ 완전 일치" if r["full_strict"] == 1 else "✗ 불일치"
    print(f"\n[예시 {i+1}] {mark}")
    print(f"  문장:        {r['sentence']}")
    print(f"  gold:        {r['gold']}")
    print(f"  pred (원본): {r['pred_raw']}")
    print(f"  pred (정제): {r['pred_normalized']}")
    print(f"  Strict — seg: {r['seg_strict']}  full: {r['full_strict']}")
    print(f"  Degree — seg: {r['seg_degree']:.4f}  full: {r['full_degree']:.4f}")


완료!

=== [Only Inference (Gemma)] 결과 요약 (n=10) ===
  parse_fail: 0 (0.0%)
  [Strict]  seg: 100.00%   full: 100.00%
  [Degree]  seg: 100.00%   full: 100.00%
=== 오류 분석 ===
  오류 문장 수:  0 / 10 (0.0%)
  [A] 이형태 후보: 0건 (집계 안 함, 질적 분석용)
  [B] 분절 오류:   0개 문장 (0.0%)

=== 실제 추론 예시 (3개) ===

[예시 1] ✓ 완전 일치
  문장:        [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:        [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  pred (원본): [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  pred (정제): [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  Strict — seg: 1  full: 1
  Degree — seg: 1.0000  full: 1.0000

[예시 2] ✓ 완전 일치
  문장:        [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:        [/SS 제

---
## 결과 2: Only Inference (GPT)

- 상용 LLM(OpenAI GPT)을 few-shot 프롬프팅으로 사용한 베이스라인임
  (원 노트북: `exp1_morpheme_inference_GPT_v3.ipynb`)
- GPU 불필요. `OPENAI_API_KEY` 환경변수 또는 `.env` 파일만 있으면 실행 가능

> ⚠️ 과금 주의: 이 섹션은 OpenAI API를 실제로 호출하므로 요금이 발생할 수 있음.
> 아래 **Cell 16(GPT 추론 루프)**은 안전을 위해 기본적으로 주석 처리해 둠.
> 실행하려면 해당 셀 안내를 참고해 직접 주석을 해제할 것.


In [17]:
# ── Cell 13. OpenAI API 키 설정 ──────────────────────────────
# .env 파일에서 OPENAI_API_KEY를 로드함.
#   echo 'OPENAI_API_KEY=sk-...' > .env

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError(".env 파일 또는 환경변수에 OPENAI_API_KEY가 없습니다.")

client = OpenAI(api_key=api_key)
client.models.list()  # 연결 테스트용 (모델 목록 조회, 과금 없음)
print("OpenAI 연결: OK")

MODEL_ID_GPT = "gpt-4.1-mini"


OpenAI 연결: OK


In [18]:
# ── Cell 14. 데이터 로드 + 분할 (GPT는 GPU 토크나이저 불필요) ──
# tokenizer=None → 공백 기준 토큰 수로 근사해 동일 파이프라인 재사용함.
train_samples_gpt, val_samples_gpt, test_samples_gpt = load_filter_split(
    DEMO_JSONL_PATH, tokenizer=None, seed=SPLIT_SEED
)

system_message_gpt = get_system_text(train_samples_gpt[0])
SYSTEM_MSG_GPT      = {"role": "system", "content": system_message_gpt}

# ── Few-shot 예시 구성 (Gemma 섹션과 동일한 시드·방식) ──────────
N_SHOTS_GPT = min(3, len(train_samples_gpt))
random.seed(SHOT_SEED)
few_shot_examples_gpt = random.sample(train_samples_gpt, N_SHOTS_GPT)

few_shot_messages_gpt = []
for ex in few_shot_examples_gpt:
    few_shot_messages_gpt.append({"role": "user",      "content": get_user_text(ex)})
    few_shot_messages_gpt.append({"role": "assistant", "content": get_assistant_text(ex)})

print(f"few-shot 예시 수: {len(few_shot_examples_gpt)}")
print(f"few-shot 예시: {few_shot_examples_gpt}")


원본 샘플 수: 100
필터링 후: 100개 (p95=24.0 기준)
train: 80 | val: 10 | test: 10
few-shot 예시 수: 3
few-shot 예시: [{'messages': [{'role': 'system', 'content': "너는 한국어 형태소 분석 전문가다. 아래의 태그 목록과 분절 규칙을 참고해 문장에 대한 형태소 분석을 수행하라.\n\n## 태그 세트\n[체언] NNG 일반명사 | NNP 고유명사 | NNB 의존명사 | NP 대명사 | NR 수사\n[용언] VV 동사 | VA 형용사 | VX 보조용언 | VCP 긍정지정사 | VCN 부정지정사\n[수식언] MMA 성상관형사 | MMD 지시관형사 | MMN 수관형사 | MAG 일반부사 | MAJ 접속부사\n[독립언] IC 감탄사\n[관계언] JKS 주격 | JKC 보격 | JKG 관형격 | JKO 목적격 | JKB 부사격 | JKV 호격 | JKQ 인용격 | JX 보조사 | JC 접속조사\n[의존형태] EP 선어말어미 | EF 종결어미 | EC 연결어미 | ETN 명사형전성어미 | ETM 관형형전성어미 | XPN 체언접두사 | XSN 명사파생접미사 | XSV 동사파생접미사 | XSA 형용사파생접미사 | XR 어근\n[기호] SF 마침표/물음표/느낌표 | SP 쉼표류 | SS 따옴표/괄호 | SE 줄임표 | SO 붙임표 | SW 기타기호\n[기타] SL 외국어 | SH 한자 | SN 숫자 | NA 분석불능 | NF 명사추정 | NV 용언추정\n\n## 분절 규칙\n반드시 형태소 단위로 분리하라. 어절 단위로 묶어서 출력하지 말라.\n본 분석은 엄밀히 말해 '형태소' 차원이 아닌 '형태' 차원의 분석이므로 이형태를 최대한 반영한다.\n예) 잡아서 → 잡/VV 아서/EC | 먹어서 → 먹/VV 어서/EC\n\n## 출력 형식\n형태소/태그 형태소/태그 ...\n예) 만나/VV 았/EP 습니다/EF ./SF\n    "}, {'role': 'user', 'content': '[제

In [19]:
# ── Cell 15. GPT API 호출 함수 ────────────────────────────────
# 함수 정의만 하는 셀이라 실행 자체로는 과금 없음. 실제 호출은 Cell 16에서 발생함.
import time

def call_gpt(sentence: str, max_retries: int = 3) -> str:
    """
    단일 문장을 받아 GPT에 형태소 분석을 요청하고 결과 반환.
    재시도(retry) 로직 포함: 네트워크 오류나 rate limit 에러 대응.
    """
    messages = (
        [SYSTEM_MSG_GPT]
        + few_shot_messages_gpt
        + [{"role": "user", "content": sentence}]
    )

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_ID_GPT,
                messages=messages,
                temperature=0,  # 0 = greedy (재현성 최대)
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f"  [오류] attempt {attempt+1}: {e} → {wait}초 후 재시도")
            time.sleep(wait)

    print(f"  [실패] {sentence[:30]}... — 빈 문자열 반환")
    return ""


In [20]:
# ── Cell 16. GPT 추론 루프 ────────────────────────────────────
# ⚠️ 과금 경고: 아래 블록은 OpenAI GPT API를 N_GPT_DEMO회 실제로 호출함.
# 호출할 때마다 OpenAI 계정에 요금이 부과될 수 있음.
# 안전을 위해 기본적으로 전체 주석 처리해 둠.
# 실행 의사가 확실하면 아래 블록의 '#'을 전부 해제한 뒤 실행할 것.
# [DEMO] 비용 절감을 위해 test_samples_gpt 앞부분 일부만 사용하도록 이미 축소해 둠.

# from tqdm import tqdm
#
# N_GPT_DEMO = min(5, len(test_samples_gpt))
# results_gpt = []
#
# for sample in tqdm(test_samples_gpt[:N_GPT_DEMO], desc="GPT 추론"):
#     sentence        = get_user_text(sample)
#     pred_raw        = call_gpt(sentence)
#     pred_normalized = normalize_pred(pred_raw)
#     gold_pairs      = parse_assistant_output(get_assistant_text(sample))
#     pred_pairs      = parse_assistant_output(pred_normalized)
#     m               = compute_metrics(gold_pairs, pred_pairs)
#
#     results_gpt.append({
#         "sentence": sentence, "gold": get_assistant_text(sample),
#         "pred_raw": pred_raw, "pred_normalized": pred_normalized,
#         "parse_success": len(pred_pairs) > 0, "truncated": False,
#         **{k: m[k] for k in ["seg_strict", "full_strict", "seg_lcs", "full_lcs", "total", "seg_degree", "full_degree"]},
#     })
#     time.sleep(0.5)  # RPM 회피용 대기
#
# print("완료!")
# summary_gpt = show_summary("Only Inference (GPT)", results_gpt)
# run_error_analysis(results_gpt)
#
# # ── 실제 추론 예시 확인 ────────────────────────────────────────
# # 집계된 숫자만으로는 GPT가 실제로 어떻게 틀리는지 보기 어려우므로,
# # gold(정답)와 pred(모델 출력)를 나란히 몇 개 직접 출력해 확인함.
# N_EXAMPLES = min(3, len(results_gpt))
# print("\n" + "=" * 60)
# print(f"=== 실제 추론 예시 ({N_EXAMPLES}개) ===")
# print("=" * 60)
# for i, r in enumerate(results_gpt[:N_EXAMPLES]):
#     mark = "✓ 완전 일치" if r["full_strict"] == 1 else "✗ 불일치"
#     print(f"\n[예시 {i+1}] {mark}")
#     print(f"  문장:        {r['sentence']}")
#     print(f"  gold:        {r['gold']}")
#     print(f"  pred (원본): {r['pred_raw']}")
#     print(f"  pred (정제): {r['pred_normalized']}")
#     print(f"  Strict — seg: {r['seg_strict']}  full: {r['full_strict']}")
#     print(f"  Degree — seg: {r['seg_degree']:.4f}  full: {r['full_degree']:.4f}")


GPT 추론: 100%|██████████| 5/5 [00:17<00:00,  3.56s/it]

완료!

=== [Only Inference (GPT)] 결과 요약 (n=5) ===
  parse_fail: 0 (0.0%)
  [Strict]  seg: 0.00%   full: 0.00%
  [Degree]  seg: 75.83%   full: 75.83%
=== 오류 분석 ===
  오류 문장 수:  5 / 5 (100.0%)
  [A] 이형태 후보: 0건 (집계 안 함, 질적 분석용)
  [B] 분절 오류:   5개 문장 (100.0%)

=== 실제 추론 예시 (3개) ===

[예시 1] ✗ 불일치
  문장:        [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:        [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  pred (원본): [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 기/ETM 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 었/EP 다/EF
  pred (정제): [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 기/ETM 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 었/EP 다/EF
  Strict — seg: 0  full: 0
  Degree — seg: 0.9583  full: 0.9583

[예시 2] ✗ 불일치
  문장:        [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 

---
## 결과 3: SFT (Gemma)

- 로컬 Gemma 모델을 QLoRA로 SFT한 뒤 추론한 결과임
  (원 노트북: `exp1_morpheme_SFT_v4.ipynb`)
- GPU + HuggingFace 토큰 필요. `결과 1` 섹션에서 이미 로드한 `tokenizer`를 재사용하고,
  학습용 모델은 별도로 다시 로드함(추론 전용 모델과 분리 목적)

> [DEMO] 데이터가 동일 문장 100개 반복이라 학습 자체는 사실상 의미 없으나,
> LoRA 설정 → SFTTrainer 구성 → 학습 실행 → 저장 → 추론까지의 **코드 경로**를 그대로 보여줌.


In [21]:
# ── Cell 17. 모델/토크나이저 로드 (4-bit, 학습용) ─────────────
from transformers import AutoTokenizer, AutoModelForImageTextToText, BitsAndBytesConfig

if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    torch_dtype = torch.bfloat16
else:
    torch_dtype = torch.float16

bnb_config_sft = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_quant_storage=torch_dtype,
)

tokenizer_sft = AutoTokenizer.from_pretrained(model_id)
try:
    model_sft = AutoModelForImageTextToText.from_pretrained(
        model_id,
        dtype=torch_dtype,
        device_map={"": 0} if torch.cuda.is_available() else "cpu",
        quantization_config=bnb_config_sft if torch.cuda.is_available() else None,
    )
    print(f"SFT용 모델 로드 완료: {model_id}")
except (PermissionError, OSError) as e:
    print(f"❌ 모델 로드 실패 (캐시 폴더 권한 오류로 추정됨): {e}")
    print("   → readme/README.md의 'HuggingFace 캐시 권한 문제' 항목을 참고하세요.")
    raise


Loading weights: 100%|██████████| 1951/1951 [00:00<00:00, 1999.56it/s]


SFT용 모델 로드 완료: google/gemma-4-E2B-it


In [22]:
# ── Cell 18. LoRA 설정 ────────────────────────────────────────
# Google 공식 코드 기반.
# target_modules="all-linear": 지원 가능한 Linear 레이어 자동 탐색
# modules_to_save: lm_head, embed_tokens를 full precision으로 학습
# ensure_weight_tying: lm_head와 embed_tokens의 weight tying 유지

from peft import LoraConfig

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=16,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"],
    ensure_weight_tying=True,
)
print("LoRA config 정의 완료")


LoRA config 정의 완료


In [23]:
# ── Cell 19. SFT 데이터셋 구성 + 서브샘플링 ────────────────────
# [DEMO] 원 실험은 N_TRAIN=10,000 기준이었으나, 데모 데이터는 100개뿐이므로
# min()으로 안전하게 개수를 맞춤. (기존 실험의 min() 가드 패턴 재사용)
from datasets import Dataset

N_TRAIN = min(10_000, len(train_samples))
N_VAL   = min(max(1, N_TRAIN // 8), len(val_samples))
N_TEST  = min(max(1, N_TRAIN // 8), len(test_samples))

random.seed(SPLIT_SEED)
train_sampled = random.sample(train_samples, N_TRAIN)
val_sampled   = random.sample(val_samples,   N_VAL)
test_sampled  = random.sample(test_samples,  N_TEST)

train_dataset = Dataset.from_list(train_sampled)
val_dataset   = Dataset.from_list(val_sampled)

print(f"train: {len(train_dataset)} | val: {len(val_dataset)} | test: {len(test_sampled)}")


train: 80 | val: 10 | test: 10


In [24]:
# ── Cell 20. 학습 설정 + 실행 ──────────────────────────────────
# [DEMO] 원 실험은 epoch=3, max_length=1024였으나, 데모는 데이터가 작으므로
# epoch=1, max_length를 짧게 조정해 빠르게 동작 확인만 함.
from trl import SFTConfig, SFTTrainer

output_dir = "./checkpoints/morpheme_sft_demo"

args = SFTConfig(
    output_dir=output_dir,
    max_length=512,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="adamw_torch_fused",
    learning_rate=5e-5,
    max_grad_norm=0.3,
    lr_scheduler_type="constant",
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="steps",
    eval_steps=10,
    save_total_limit=1,
    bf16=(model_sft.dtype == torch.bfloat16),
    fp16=(model_sft.dtype == torch.float16),
    push_to_hub=False,
    report_to="none",
    remove_unused_columns=False,
    dataset_kwargs={
        "add_special_tokens": False,
        "append_concat_token": True,
    },
)

# ── 멀티 GPU(nn.DataParallel) 자동 래핑 방지 ────────────────────
# Trainer는 self.args.n_gpu > 1이면 모델을 torch.nn.DataParallel로 자동으로 감쌈.
# 하지만 model_sft는 이미 4bit 양자화 + device_map={"": 0}로 GPU 0 한 곳에만
# 고정 배치되어 있어, DataParallel로 복제를 시도하면 rotary embedding 연산
# 등에서 CUDA illegal memory access가 발생함 (GPU가 여러 장 보이는 멀티 GPU 환경 전반에 해당).
# → args.n_gpu를 1로 강제 고정함.
#   싱글 GPU 환경에서는 어차피 n_gpu가 이미 1이므로 이 줄은 영향 없음
#   (두 환경 모두에서 안전하게 동작함).
args._n_gpu = 1
print(f"Trainer가 사용할 GPU 수: {args.n_gpu}")

trainer = SFTTrainer(
    model=model_sft,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    processing_class=tokenizer_sft,
)

trainer.train()
print("SFT 학습 완료 (demo)")


Trainer가 사용할 GPU 수: 1


Tokenizing eval dataset: 100%|██████████| 10/10 [00:00<00:00, 1576.33 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss
10,4.999176,2.036502
20,2.605170,1.028957
30,0.977407,0.297043
40,0.113267,0.019381


SFT 학습 완료 (demo)


In [25]:
# ── Cell 21. 모델 저장 ────────────────────────────────────────
from pathlib import Path
from datetime import datetime
import zoneinfo

timestamp   = datetime.now(zoneinfo.ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d_%H:%M:%S")
save_dir    = f"./models/{timestamp}_gemma_sft_demo"

Path(save_dir).mkdir(parents=True, exist_ok=True)
trainer.save_model(save_dir)
print(f"모델 저장 완료: {save_dir}")


모델 저장 완료: ./models/2026-08-09_23:23:08_gemma_sft_demo


In [26]:
# ── Cell 22. SFT 모델 추론 파라미터 설정 ────────────────────
tokenizer_sft.padding_side = "left"
tokenizer_sft.pad_token    = tokenizer_sft.eos_token

token_lengths_sft = [len(tokenizer_sft.encode(get_assistant_text(s))) for s in test_sampled]
INFER_MAX_TOKENS_SFT = int(max(token_lengths_sft) * 1.5)
INFER_REP_PENALTY_SFT = 1.1
INFER_BATCH_SIZE_SFT  = min(4, len(test_sampled))

print(f"INFER_MAX_TOKENS_SFT: {INFER_MAX_TOKENS_SFT}")


INFER_MAX_TOKENS_SFT: 135


In [32]:
# ── Cell 23. SFT 모델 배치 추론 루프 ─────────────────────────
# trainer.model(=model_sft, LoRA adapter가 이미 적용된 상태)을 그대로 사용함.
system_message_sft = get_system_text(test_sampled[0])
SYSTEM_MSG_SFT      = {"role": "system", "content": system_message_sft}

model_sft.eval()
results_sft = []

for batch_start in range(0, len(test_sampled), INFER_BATCH_SIZE_SFT):
    batch_samples = test_sampled[batch_start:batch_start + INFER_BATCH_SIZE_SFT]

    batch_messages = [
        [SYSTEM_MSG_SFT, {"role": "user", "content": get_user_text(s)}]
        for s in batch_samples
    ]

    inputs = tokenizer_sft.apply_chat_template(
        batch_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
        padding=True,
    ).to(DEVICE)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        outputs = model_sft.generate(
            **inputs,
            max_new_tokens=INFER_MAX_TOKENS_SFT,
            do_sample=False,
            use_cache=True,
            top_p=None,
            top_k=None,
            repetition_penalty=INFER_REP_PENALTY_SFT,
        )

    for j, sample in enumerate(batch_samples):
        pred_tokens    = outputs[j][input_len:]
        truncated      = pred_tokens.shape[-1] >= INFER_MAX_TOKENS_SFT

        pred_raw        = tokenizer_sft.decode(pred_tokens, skip_special_tokens=True).strip()
        pred_normalized = normalize_pred(pred_raw)
        gold_pairs      = parse_assistant_output(get_assistant_text(sample))
        pred_pairs      = parse_assistant_output(pred_normalized)
        m               = compute_metrics(gold_pairs, pred_pairs)

        results_sft.append({
            "sentence": get_user_text(sample), "gold": get_assistant_text(sample),
            "pred_raw": pred_raw, "pred_normalized": pred_normalized,
            "parse_success": len(pred_pairs) > 0, "truncated": truncated,
            **{k: m[k] for k in ["seg_strict", "full_strict", "seg_lcs", "full_lcs", "total", "seg_degree", "full_degree"]},
        })

print("완료!")
summary_sft = show_summary("SFT (Gemma)", results_sft)
run_error_analysis(results_sft)

# ── 실제 추론 예시 확인 ────────────────────────────────────────
# 집계된 숫자만으로는 SFT 모델이 실제로 어떻게 틀리는지 보기 어려우므로,
# gold(정답)와 pred(모델 출력)를 나란히 몇 개 직접 출력해 확인함.
N_EXAMPLES = min(3, len(results_sft))
print("\n" + "=" * 60)
print(f"=== 실제 추론 예시 ({N_EXAMPLES}개) ===")
print("=" * 60)
for i, r in enumerate(results_sft[:N_EXAMPLES]):
    mark = "✓ 완전 일치" if r["full_strict"] == 1 else "✗ 불일치"
    print(f"\n[예시 {i+1}] {mark}")
    print(f"  문장:        {r['sentence']}")
    print(f"  gold:        {r['gold']}")
    print(f"  pred (원본): {r['pred_raw']}")
    print(f"  pred (정제): {r['pred_normalized']}")
    print(f"  Strict — seg: {r['seg_strict']}  full: {r['full_strict']}")
    print(f"  Degree — seg: {r['seg_degree']:.4f}  full: {r['full_degree']:.4f}")


완료!

=== [SFT (Gemma)] 결과 요약 (n=10) ===
  parse_fail: 0 (0.0%)
  [Strict]  seg: 0.00%   full: 0.00%
  [Degree]  seg: 25.00%   full: 12.50%
=== 오류 분석 ===
  오류 문장 수:  10 / 10 (100.0%)
  [A] 이형태 후보: 0건 (집계 안 함, 질적 분석용)
  [B] 분절 오류:   10개 문장 (100.0%)

=== 실제 추론 예시 (3개) ===

[예시 1] ✗ 불일치
  문장:        [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:        [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  pred (원본): 제주/NNP 서울/NNP "세계환경수도/NN 일반명사 조성/NND 위해/EC 10개년/NM 실천계획/NND 만들겠/VV 다/CP 김태환/NNP 지사/NNS 밝혀/VV ./SF
  pred (정제): 제주/NNP 서울/NNP "세계환경수도/NN 일반명사 조성/NND 위해/EC 10개년/NM 실천계획/NND 만들겠/VV 다/CP 김태환/NNP 지사/NNS 밝혀/VV ./SF
  Strict — seg: 0  full: 0
  Degree — seg: 0.2500  full: 0.1250

[예시 2] ✗ 불일치
  문장:        [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:        [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/S

---
## 결과 4: Kiwi (규칙/통계 기반 베이스라인)

- LLM이 아닌 전통적 형태소 분석기 Kiwi와의 비교임. GPU 불필요
  (원 노트북: `exp1_morpheme_kiwi_v3.ipynb`)


In [33]:
# ── Cell 24. Kiwi 설치 및 로드 ────────────────────────────────
# kiwipiepy: 한국어 형태소 분석기
# - 규칙 기반 + 통계 기반 하이브리드 방식
# - 자체 POS 태그셋 사용 → 모두의 말뭉치(NIKL) 태그셋과 일부 불일치
# - 베이스라인 목적: LLM 대비 규칙 기반 분석기의 성능 하한선 확인

from kiwipiepy import Kiwi
import kiwipiepy

kiwi = Kiwi()
print(f"kiwipiepy 버전: {kiwipiepy.__version__}")
print("Kiwi 로드 완료")


kiwipiepy 버전: 0.23.1
Kiwi 로드 완료


In [34]:
# ── Cell 25. 데이터 로드 + 분할 (Kiwi는 GPU 토크나이저 불필요) ──
train_samples_kiwi, val_samples_kiwi, test_samples_kiwi = load_filter_split(
    DEMO_JSONL_PATH, tokenizer=None, seed=SPLIT_SEED
)


원본 샘플 수: 100
필터링 후: 100개 (p95=24.0 기준)
train: 80 | val: 10 | test: 10


In [35]:
# ── Cell 26. Kiwi 태그 → NIKL 태그 매핑 + 정규화 ─────────────
#
# [태그 매핑: Kiwi → NIKL]
# Kiwi는 세종 태그 기반이나 NIKL과 일부 불일치함:
#   MM  → NIKL의 MMA/MMD/MMN 세분 불가 → MM으로 통일
#   XSM → NIKL 미존재 → None ('??' 처리)
#   SSO/SSC → NIKL의 SS로 통합
#   SB, UN, W_*, Z_*, USER0~4 → NIKL 미존재 → None
#
# [gold(NIKL) 태그 정규화]
# Kiwi의 MM은 NIKL의 MMA(성상)/MMD(지시)/MMN(수) 모두에 대응함.
# 평가 시 gold 태그도 MM으로 통일하여 Kiwi 출력과 비교 가능하게 함:
#   MMA → MM
#   MMD → MM
#   MMN → MM
#
# [종성자 정규화]
# Kiwi는 종성 자음을 독립 자모(ᆫ ᆯ ᆸ)로 출력하나,
# NIKL은 초성 자모(ㄴ ㄹ ㅂ)로 표기함.
# 유니코드상 다른 문자이나 언어학적으로 동일한 형태소이므로
# Kiwi 출력에 정규화를 적용해 일치로 처리함.
#
# [이형태 처리 방침]
# 아/어, ㄴ/은, ㄹ/을 등 음운론적 이형태는 정규화하지 않고 오류로 처리함.
# → Kiwi와 NIKL의 기저형 선택 기준 차이로 인해 성능이 과소평가될 수 있음.

KIWI_TO_NIKL = {
    # 체언
    "NNG": "NNG", "NNP": "NNP", "NNB": "NNB", "NR": "NR", "NP": "NP",
    # 용언
    "VV": "VV", "VA": "VA", "VX": "VX", "VCP": "VCP", "VCN": "VCN",
    # 관형사 (Kiwi는 MM 단일 태그만 사용 → gold도 정규화해 비교)
    "MM": "MM",
    # 부사
    "MAG": "MAG", "MAJ": "MAJ",
    # 감탄사
    "IC": "IC",
    # 조사
    "JKS": "JKS", "JKC": "JKC", "JKG": "JKG", "JKO": "JKO", "JKB": "JKB",
    "JKV": "JKV", "JKQ": "JKQ", "JX": "JX", "JC": "JC",
    # 어미
    "EP": "EP", "EF": "EF", "EC": "EC", "ETN": "ETN", "ETM": "ETM",
    # 접두사·접미사
    "XPN": "XPN", "XSN": "XSN", "XSV": "XSV", "XSA": "XSA",
    "XSM": None,  # 부사 파생 접미사 → NIKL 부재
    "XR": "XR",
    # 기호
    "SF": "SF", "SP": "SP", "SS": "SS",
    "SSO": "SS", "SSC": "SS",  # 여닫는 괄호 → SS로 통합
    "SE": "SE", "SO": "SO", "SW": "SW",
    "SB": None,  # 순서 있는 글머리 → NIKL 부재
    "SL": "SL", "SH": "SH", "SN": "SN",
    # Kiwi 전용: 분석 불능 및 웹 특수 표현
    "UN": None, "W_URL": None, "W_EMAIL": None, "W_HASHTAG": None,
    "W_MENTION": None, "W_SERIAL": None, "W_EMOJI": None,
    # Kiwi 전용: 특수 형태
    "Z_CODA": None, "Z_SIOT": None,
    # 사용자 정의 태그
    "USER0": None, "USER1": None, "USER2": None, "USER3": None, "USER4": None,
}

NIKL_NORMALIZE = {
    "MMA": "MM", "MMD": "MM", "MMN": "MM",
}

JONGSEONG_TO_CHOSEONG = str.maketrans(
    "ᆨᆩᆪᆫᆬᆭᆮᆯᆰᆱᆲᆳᆴᆵᆶᆷᆸᆹᆺᆻᆼᆽᆾᆿᇀᇁᇂ",
    "ㄱㄲㄳㄴㄵㄶㄷㄹㄺㄻㄼㄽㄾㄿㅀㅁㅂㅄㅅㅆㅇㅈㅊㅋㅌㅍㅎ"
)

def normalize_form(form: str) -> str:
    """Kiwi 출력의 종성 자모 → NIKL 표기의 초성 자모로 변환."""
    return form.translate(JONGSEONG_TO_CHOSEONG)

def kiwi_tag_to_nikl(tag: str) -> str:
    """Kiwi 태그 → NIKL 태그 변환. 불규칙 활용 접미사(-R/-I) 제거 후 매핑."""
    base = tag.split("-")[0]
    nikl = KIWI_TO_NIKL.get(base)
    return nikl if nikl is not None else "??"

def normalize_gold_pairs(gold_pairs: list) -> list:
    """gold(NIKL) 형태소 쌍 리스트에 태그 정규화 적용 (MMA/MMD/MMN → MM)."""
    return [(form, NIKL_NORMALIZE.get(tag, tag)) for form, tag in gold_pairs]

def kiwi_to_pairs(sentence: str) -> list:
    """문장을 Kiwi로 분석하여 (form, nikl_tag) 쌍 리스트로 반환."""
    tokens = kiwi.tokenize(sentence)
    return [
        (normalize_form(tok.form), kiwi_tag_to_nikl(str(tok.tag)))
        for tok in tokens
    ]


In [37]:
# ── Cell 27. Kiwi 베이스라인 평가 ────────────────────────────
results_kiwi = []

for sample in test_samples_kiwi:
    sentence   = get_user_text(sample)
    gold_pairs = normalize_gold_pairs(parse_assistant_output(get_assistant_text(sample)))
    pred_pairs = kiwi_to_pairs(sentence)
    m          = compute_metrics(gold_pairs, pred_pairs)

    results_kiwi.append({
        "sentence": sentence, "gold": get_assistant_text(sample),
        "pred_normalized": " ".join(f"{f}/{t}" for f, t in pred_pairs),
        "parse_success": True, "truncated": False,
        **{k: m[k] for k in ["seg_strict", "full_strict", "seg_lcs", "full_lcs", "total", "seg_degree", "full_degree"]},
    })

print("완료!")
summary_kiwi = show_summary("Kiwi", results_kiwi)
run_error_analysis(results_kiwi)

# ── 실제 추론 예시 확인 ────────────────────────────────────────
# 집계된 숫자만으로는 Kiwi가 실제로 어떻게 틀리는지 보기 어려우므로,
# gold(정답)와 pred(Kiwi 출력)를 나란히 몇 개 직접 출력해 확인함.
# Kiwi는 LLM이 아니라 규칙/통계 기반 분석기라 "원본 출력"이라는 개념이 없으므로
# pred_raw 없이 pred_normalized만 표시함.
N_EXAMPLES = min(3, len(results_kiwi))
print("\n" + "=" * 60)
print(f"=== 실제 추론 예시 ({N_EXAMPLES}개) ===")
print("=" * 60)
for i, r in enumerate(results_kiwi[:N_EXAMPLES]):
    mark = "✓ 완전 일치" if r["full_strict"] == 1 else "✗ 불일치"
    print(f"\n[예시 {i+1}] {mark}")
    print(f"  문장:  {r['sentence']}")
    print(f"  gold:  {r['gold']}")
    print(f"  pred:  {r['pred_normalized']}")
    print(f"  Strict — seg: {r['seg_strict']}  full: {r['full_strict']}")
    print(f"  Degree — seg: {r['seg_degree']:.4f}  full: {r['full_degree']:.4f}")


완료!

=== [Kiwi] 결과 요약 (n=10) ===
  parse_fail: 0 (0.0%)
  [Strict]  seg: 0.00%   full: 0.00%
  [Degree]  seg: 95.83%   full: 95.83%
=== 오류 분석 ===
  오류 문장 수:  10 / 10 (100.0%)
  [A] 이형태 후보: 10건 (집계 안 함, 질적 분석용)
  [B] 분절 오류:   0개 문장 (0.0%)

=== 실제 추론 예시 (3개) ===

[예시 1] ✗ 불일치
  문장:  [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:  [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  pred:  [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 어/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  Strict — seg: 0  full: 0
  Degree — seg: 0.9583  full: 0.9583

[예시 2] ✗ 불일치
  문장:  [제주·서울] "세계환경수도 조성위해 10개년 실천계획 만들겠다" 김태환 지사 밝혀
  gold:  [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 위하/VV 아/EC 10/SN 개년/NNB 실천/NNG 계획/NNG 만들/VV 겠/EP 다/EF "/SS 김태환/NNP 지사/NNG 밝히/VV 어/EF
  pred:  [/SS 제주/NNP ·/SP 서울/NNP ]/SS "/SS 세계/NNG 환경/NNG 수도/NNG 조성/NNG 

---
## 종합 비교

- 4가지 방법의 결과를 하나의 표로 모아 비교함
- 실행하지 않은 섹션이 있으면 해당 `summary_*` 변수가 없어 자동으로 제외됨

> [DEMO] 데이터가 동일 문장의 반복이므로 수치 자체보다는
> **4개 파이프라인이 모두 동일한 형식으로 결과를 산출한다는 것**을 확인하는 데 의의가 있음.


In [ ]:
# ── Cell 28. 4가지 결과 종합 비교표 ──────────────────────────
# 실제 실험 결과와는 다르며, 작동 확인용 데모 결과임.
# 동일한 문장에 대한 비정상적인 반복 학습으로 인해 SFT의 결과가 좋지 않을 수 있음.
import pandas as pd

all_summaries = []
for name in ["summary_gemma_inference", "summary_gpt", "summary_sft", "summary_kiwi"]:
    if name in dir():
        all_summaries.append(globals()[name])

if all_summaries:
    df_compare = pd.DataFrame(all_summaries)
    print(df_compare.to_string(index=False))
else:
    print("실행된 결과가 없습니다. 위 4개 섹션 중 하나 이상을 먼저 실행하세요.")


                method  n  parse_fail  seg_strict  full_strict  seg_degree  full_degree
Only Inference (Gemma) 10           0       100.0        100.0      100.00       100.00
  Only Inference (GPT)  5           0         0.0          0.0       75.83        75.83
           SFT (Gemma) 10           0         0.0          0.0       25.00        12.50
                  Kiwi 10           0         0.0          0.0       95.83        95.83
